Compute hitting time from cumulative probability from traversal models

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from utils.config import DATA_DIR, FIG_DIR

result_dir = FIG_DIR / 'network'
result_dir.mkdir(parents=True, exist_ok=True)

neuron_info = pd.read_pickle(DATA_DIR / 'neuron_info_ol.pkl')

# hitting time from Bayesian traversal model

In [ ]:
# load bayesian traversal results
i = 3
res = pd.read_pickle(Path(result_dir, 'scan_all', f'btModel_ethr5_maxW_{i}.pkl'))
summ = pd.read_pickle(Path(result_dir, 'scan_all', f'btModel_ethr5_summary_maxW_{i}.pkl'))

In [ ]:
m = res['cmf']
# convert m to a matrix
m= np.stack(m.values)

In [ ]:
# compute hitting time
hitting_prob = np.diff(m, axis=1)
hitting_time = np.dot(hitting_prob, np.arange(2, 11))

In [ ]:
# set hittig time to 1 for seeds
seeds = neuron_info[neuron_info['type'].str.contains(r'(^L[1-3]{1})|(^R[78]{1})|(HBeyelet)')] ['bodyId'].values
hitting_time_seeds = res['node'].isin(seeds).astype(int)

In [ ]:
hitting_time = hitting_time_seeds + hitting_time 

# 0 means not reachable

In [ ]:
# add res['node'] 
hitting_time = pd.DataFrame({'bodyId': res['node'], 'hitting_time': hitting_time})

## compare with level assignment

In [ ]:
# layer median
df = pd.merge(neuron_info, summ['layer_median'], left_on='bodyId', right_index=True, how='left')
inst_layer = df.groupby('instance').agg({'layer_median': 'mean', 'consensusNt': 'first', 'main_groups':'first'}).reset_index()

# hitting time
df = pd.merge(neuron_info, hitting_time, left_on='bodyId', right_on='bodyId', how='left')
inst_ht = df.groupby('instance').agg({'hitting_time': 'mean', 'consensusNt': 'first', 'main_groups':'first'}).reset_index()

df = pd.merge(inst_layer, inst_ht, on='instance', suffixes=('_bt', '_ht'))
# compute layer_median - layer_min
df['diff'] = df['layer_median'] - df['hitting_time']

In [ ]:
# plot
import matplotlib.pyplot as plt

# hist of df['diff']
fig = plt.figure()
plt.hist(df[df['layer_median'].notna() & (df['layer_median'] != -1)]['diff'], bins=100)
plt.xlabel('Layer_median - hitting_time')
plt.ylabel('Count')
plt.title('Histogram of Layer_median - hitting_time')
plt.show()